# 🔍 Model Interpretation & Evaluation
## In-Depth Validation of Disaster Frequency Predictions

**This notebook provides rigorous model validation through:**
- Explicit Train / Validation / Test split on the global trend series
- GridSearchCV hyperparameter tuning for KNN Regression
- Residual analysis — the primary diagnostic for regression models
- Feature importance via Linear Regression coefficients
- Final conclusions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 📂 Step 0: Load Data from Previous Notebooks

In [2]:
# Load prediction results from notebook 2
comparison_df = pd.read_csv('/content/complete_prediction_results.csv')
agg = pd.read_csv('/content/processed_agg.csv')

# Load global aggregated data for trend analysis
import pandas as pd
df_raw = pd.read_csv('/content/_EmergencyEventsDatabase-CountryProfiles_emdat-country-profiles_2023_04_06.csv', sep=';')
agg_global = df_raw.groupby('Year').size().reset_index(name='Disaster_Count')

print(f"Comparison df: {len(comparison_df)} combinations")
print(f"Agg data     : {len(agg)} rows")
print(f"Global trend : {len(agg_global)} years")

## ✂️ Step 1: Train / Validation / Test Split
We demonstrate an explicit data split on the **global yearly disaster trend** (total events per year, all countries combined).

**Split: 70% train | 15% validation | 15% test**  
- `shuffle=False` — this is a time series; future data must not leak into training  
- The test set represents the most recent years and is touched **only once** for final evaluation

**Why the global series?** The per-combination modeling in Notebook 2 uses an implicit 70/30 split inside each series. Here we use the aggregated global series to provide a clean, visualizable demonstration with enough data points for a meaningful three-way split.

In [3]:
X_all = agg_global['Year'].values.reshape(-1, 1)
y_all = agg_global['Disaster_Count'].values

X_temp, X_test_split, y_temp, y_test_split = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, shuffle=False)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, shuffle=False)

print("=== DATASET SPLIT (Global Trend) ===")
print(f"Total data    : {len(X_all)} years")
print(f"Training set  : {len(X_train_split)} ({len(X_train_split)/len(X_all)*100:.1f}%) — Year {X_train_split.min()} to {X_train_split.max()}")
print(f"Validation set: {len(X_val_split)} ({len(X_val_split)/len(X_all)*100:.1f}%) — Year {X_val_split.min()} to {X_val_split.max()}")
print(f"Test set      : {len(X_test_split)} ({len(X_test_split)/len(X_all)*100:.1f}%) — Year {X_test_split.min()} to {X_test_split.max()}")

plt.figure(figsize=(12, 4))
plt.scatter(X_train_split, y_train_split, color='#2E86C1', label=f'Training ({len(X_train_split)})', s=30)
plt.scatter(X_val_split,   y_val_split,   color='#F39C12', label=f'Validation ({len(X_val_split)})', s=30)
plt.scatter(X_test_split,  y_test_split,  color='#E74C3C', label=f'Test ({len(X_test_split)})', s=30)
plt.title('Train-Validation-Test Split — Global Disaster Trend by Year')
plt.xlabel('Year')
plt.ylabel('Event Count')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('train_val_test_split.png', dpi=150)
plt.show()

> ✅ **Result**: The global disaster time series spans **124 years** (1900–2023). Training covers the earliest ~87 years, validation the next ~19 years, and the test set the most recent ~18 years — the model is always evaluated on data it has never seen during training or tuning.

## 🔧 Step 2: Hyperparameter Tuning — GridSearchCV for KNN
We search for the best KNN hyperparameter combination using **GridSearchCV** with 3-fold cross-validation **on the training set only**.

**Parameters searched:**
- `n_neighbors`: [1, 2, 3, 4, 5, 7, 10]  
- `weights`: ['uniform', 'distance']  
- `metric`: ['euclidean', 'manhattan']

**Scoring metric:** negative MAE (minimized)  

The validation and test sets are **never used during tuning** — GridSearchCV only sees `X_train_split` / `y_train_split`.

In [4]:
pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid = {
    'knn__n_neighbors': [1, 2, 3, 4, 5, 7, 10],
    'knn__weights'    : ['uniform', 'distance'],
    'knn__metric'     : ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(pipeline_knn, param_grid, cv=3,
                           scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train_split, y_train_split)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best MAE (CV)    : {-grid_search.best_score_:.3f}")

# Evaluate model on the validation set
lr_val = LinearRegression()
lr_val.fit(X_train_split, y_train_split)
y_pred_lr_val  = lr_val.predict(X_val_split)
y_pred_knn_val = grid_search.best_estimator_.predict(X_val_split)

print(f"\n=== VALIDATION SET ===")
print(f"{'Metric':<8} {'Linear Regression':>20} {'KNN Tuned':>12}")
print("-" * 43)
for metric, fn in [('MAE', mean_absolute_error), ('RMSE', lambda a,b: np.sqrt(mean_squared_error(a,b))), ('R²', r2_score)]:
    print(f"{metric:<8} {fn(y_val_split, y_pred_lr_val):>20.3f} {fn(y_val_split, y_pred_knn_val):>12.3f}")

# Evaluate model on the test set
y_pred_lr_test  = lr_val.predict(X_test_split)
y_pred_knn_test = grid_search.best_estimator_.predict(X_test_split)

print(f"\n=== TEST SET (Final Evaluation) ===")
print(f"{'Metric':<8} {'Linear Regression':>20} {'KNN Tuned':>12}")
print("-" * 43)
for metric, fn in [('MAE', mean_absolute_error), ('RMSE', lambda a,b: np.sqrt(mean_squared_error(a,b))), ('R²', r2_score)]:
    print(f"{metric:<8} {fn(y_test_split, y_pred_lr_test):>20.3f} {fn(y_test_split, y_pred_knn_test):>12.3f}")

mae_lr_test  = mean_absolute_error(y_test_split, y_pred_lr_test)
mae_knn_test = mean_absolute_error(y_test_split, y_pred_knn_test)
print(f"\n🏆 Best model on test set: {'Linear Regression' if mae_lr_test <= mae_knn_test else 'KNN Regression (Tuned)'}")

> 📌 **Tuning Result**: The best KNN configuration is selected based on cross-validated MAE. Performance is checked on the validation set to confirm tuning helped, then evaluated once on the held-out test set for an unbiased final comparison against Linear Regression.

## 📉 Step 3: Residual Analysis
Residual plots are the primary diagnostic for regression models. A well-behaved model should produce residuals that are:
1. **Randomly scattered around zero** — no systematic bias
2. **Constant in spread** — no funnel shape (homoscedasticity)
3. **Without obvious structure** — no curve or pattern

Violations suggest the model is missing something — e.g. a non-linear relationship or an omitted variable.

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual Analysis — Linear Regression vs KNN Regression',
             fontsize=13, fontweight='bold')

residual_lr  = y_test_split - y_pred_lr_test
residual_knn = y_test_split - y_pred_knn_test

# Generate residual plot for Linear Regression
axes[0,0].scatter(y_pred_lr_test, residual_lr, color='#2E86C1', alpha=0.7, edgecolors='black', s=60)
axes[0,0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0,0].set_title('Residual Plot — Linear Regression')
axes[0,0].set_xlabel('Predicted Values')
axes[0,0].set_ylabel('Residual')
axes[0,0].grid(alpha=0.3)

# Generate residual plot for KNN Regression
axes[0,1].scatter(y_pred_knn_test, residual_knn, color='#E74C3C', alpha=0.7, edgecolors='black', s=60)
axes[0,1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0,1].set_title('Residual Plot — KNN Regression (Tuned)')
axes[0,1].set_xlabel('Predicted Values')
axes[0,1].set_ylabel('Residual')
axes[0,1].grid(alpha=0.3)

# Plot Actual vs Predicted values for Linear Regression
axes[1,0].scatter(y_test_split, y_pred_lr_test, color='#2E86C1', alpha=0.7, edgecolors='black', s=60)
min_v, max_v = y_test_split.min(), y_test_split.max()
axes[1,0].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1.5)
axes[1,0].set_title('Actual vs Predicted — Linear Regression')
axes[1,0].set_xlabel('Actual Values')
axes[1,0].set_ylabel('Predicted Values')
axes[1,0].grid(alpha=0.3)

# Plot Actual vs Predicted values for KNN Regression
axes[1,1].scatter(y_test_split, y_pred_knn_test, color='#E74C3C', alpha=0.7, edgecolors='black', s=60)
axes[1,1].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1.5)
axes[1,1].set_title('Actual vs Predicted — KNN Regression (Tuned)')
axes[1,1].set_xlabel('Actual Values')
axes[1,1].set_ylabel('Predicted Values')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('residual_plot.png', dpi=150)
plt.show()
print("✅ Residual analysis complete!")

> 📌 **Residual Interpretation**: Both models show residuals scattered around zero without a strong systematic pattern, suggesting neither model is severely misspecified for the global trend series. Some spread at higher predicted values is expected — disaster frequency in recent decades shows higher variability, likely driven by climate-related acceleration.

## 🔍 Step 4: Feature Importance — Linear Regression Coefficients
In our setup, **time** (the bin midpoint year) is the only predictor. The Linear Regression coefficient directly captures the **average rate of change** in disaster frequency per year — i.e. the trend slope.

- **Positive coefficient** → disaster frequency trending upward over time  
- **Negative coefficient** → disaster frequency trending downward  
- **Near zero** → no clear linear trend

By aggregating coefficients across all 488 combinations, we identify which disaster types are increasing fastest globally.

In [6]:
def aggregate_by_granularity(data, granularity):
    temp = data.copy()
    temp['Bin'] = (temp['Year'] // granularity) * granularity
    return temp.groupby('Bin')['Disaster_Count'].sum().reset_index()

koef_list = []
for _, row in comparison_df.iterrows():
    negara = row['Country']
    jenis  = row['Disaster Type']
    g      = row['Granularity (years)']
    subset = agg[(agg['Country'] == negara) & (agg['Disaster Type'] == jenis)]
    binned = aggregate_by_granularity(subset, g)
    if len(binned) < 4:
        continue
    X_fi = binned['Bin'].values.reshape(-1, 1)
    y_fi = binned['Disaster_Count'].values
    m    = LinearRegression()
    m.fit(X_fi, y_fi)
    koef_list.append({'Country': negara, 'Disaster Type': jenis, 'Coefficient': m.coef_[0]})

koef_df = pd.DataFrame(koef_list)

print("Top 10 Combinations - UPWARD TREND most significant:")
print(koef_df.nlargest(10, 'Coefficient')[['Country','Disaster Type','Coefficient']].to_string(index=False))

print("\nTop 10 Combinations - DOWNWARD TREND most significant:")
print(koef_df.nsmallest(10, 'Coefficient')[['Country','Disaster Type','Coefficient']].to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance — Tren Waktu per Disaster Type', fontsize=13, fontweight='bold')

avg_koef = koef_df.groupby('Disaster Type')['Coefficient'].mean().sort_values()
colors   = ['#E74C3C' if x > 0 else '#2E86C1' for x in avg_koef.values]
axes[0].barh(avg_koef.index, avg_koef.values, color=colors, edgecolor='black')
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('Average Coefficient by Disaster Type\n(Red=Increasing, Blue=Decreasing)')
axes[0].set_xlabel('Average Coefficient')

axes[1].hist(koef_df['Coefficient'], bins=30, color='#9B59B6', edgecolor='black', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Coefficient=0')
axes[1].axvline(x=koef_df['Coefficient'].mean(), color='orange', linestyle='--',
                linewidth=1.5, label=f"Mean={koef_df['Coefficient'].mean():.4f}")
axes[1].set_title('Coefficient Distribution Across All Combinations')
axes[1].set_xlabel('Coefficient')
axes[1].legend()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print("✅ Feature importance complete!")

> 📌 **Feature Importance Findings**:  
> - The distribution of coefficients is right-skewed — more combinations show positive (increasing) trends than negative, consistent with the global hypothesis.  
> - **Flood and Storm** show the highest average positive coefficients, confirming they are the fastest-growing disaster types globally.  
> - Some combinations show negative coefficients — this may reflect improved disaster prevention, reporting changes, or genuine frequency reductions in specific regions.

In [ ]:
# CELL LAUNCHER DASHBOARD STREAMLIT (5-Year Projection: 2027-2031)
!pip install streamlit pyngrok -q
!ngrok authtoken 3GXCrwYvIwvLu3Iu74ThTGP6Mod_7M24bd9UZyY34fhzjnXyk

import pandas as pd, matplotlib, matplotlib.pyplot as plt, pickle
matplotlib.use('Agg')

# 2. Generate model comparison boxplot
df_pred = pd.read_csv('/content/complete_prediction_results.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Performance Comparison: Linear Regression vs KNN Regression', fontsize=13, fontweight='bold')
for ax, (col_lr, col_knn), label in zip(axes, [('MAE_LR','MAE_KNN'), ('RMSE_LR','RMSE_KNN'), ('R2_LR','R2_KNN')], ['MAE', 'RMSE', 'R2']):
    ax.boxplot([df_pred[col_lr], df_pred[col_knn]])
    ax.set_xticklabels(['Linear Regression', 'KNN Regression'])
    ax.set_title(label)
plt.tight_layout()
plt.savefig('/content/model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('model_comparison.png generated successfully.')

# 3. Write app_colab.py
app_code = """import streamlit as st\nimport pandas as pd\nimport pickle\nimport os\nimport matplotlib\nmatplotlib.use('Agg')\nimport matplotlib.pyplot as plt\n\n# ---------------------------------------------------------------------------\n# Path resolution — works both locally and when run from any directory\n# ---------------------------------------------------------------------------\nBASE_DIR   = os.path.dirname(os.path.abspath(__file__))\nROOT_DIR   = os.path.dirname(BASE_DIR)\nDATA_FILE  = \"/content/complete_prediction_results.csv\"\nAGG_FILE   = \"/content/processed_agg.csv\"\nMODEL_FILE = \"/content/best_models.pkl\"\nIMAGE_FILE = \"/content/model_comparison.png\"\nYEAR_LIST = [2027, 2028, 2029, 2030, 2031]\n\n# ---------------------------------------------------------------------------\n# Page config\n# ---------------------------------------------------------------------------\nst.set_page_config(\n    page_title=\"Global Disaster Prediction 2027-2031\",\n    page_icon=\"🌏\",\n    layout=\"wide\",\n)\n\n# ---------------------------------------------------------------------------\n# CSS & Fonts\n# ---------------------------------------------------------------------------\nst.markdown(\"\"\"\n<style>\n@import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;700;800&family=Plus+Jakarta+Sans:wght@300;400;600;700&display=swap');\nhtml, body, [class*=\"css\"], .stMarkdown { font-family: 'Plus Jakarta Sans', sans-serif; }\nh1,h2,h3 { font-family: 'Outfit', sans-serif !important; font-weight: 700 !important; }\n.header-box {\n    background: linear-gradient(135deg, #3B82F6 0%, #8B5CF6 50%, #EC4899 100%);\n    padding: 32px; border-radius: 20px; color: white; margin-bottom: 28px;\n    box-shadow: 0 10px 25px -5px rgba(59,130,246,0.35);\n}\n.metric-card {\n    background: #FFFFFF;\n    padding: 20px 16px;\n    border-radius: 16px;\n    border: 1px solid #F1F5F9;\n    text-align: center;\n    min-height: 130px;\n    display: flex;\n    flex-direction: column;\n    align-items: center;\n    justify-content: center;\n    box-shadow: 0 4px 6px -1px rgba(0,0,0,0.05);\n    transition: all 0.3s ease;\n}\n.metric-card:hover { transform: translateY(-4px); box-shadow: 0 20px 25px -5px rgba(0,0,0,0.1); }\n.metric-icon { font-size: 26px; margin-bottom: 8px; }\n.metric-label {\n    color: #64748B; font-size: 10px; font-weight: 700;\n    text-transform: uppercase; letter-spacing: 0.08em;\n    margin-bottom: 6px; line-height: 1.3;\n}\n.metric-value {\n    color: #0F172A; font-size: 20px; font-weight: 800;\n    line-height: 1.2; word-break: break-word;\n}\n.trend-box { padding: 14px; border-radius: 12px; margin-top: 14px; border: 1px solid rgba(0,0,0,0.04); }\n.info-box {\n    background: #F8FAFC; border-left: 5px solid #6366F1;\n    padding: 18px; border-radius: 12px; margin-bottom: 18px;\n}\n</style>\n\"\"\", unsafe_allow_html=True)\n\n# ---------------------------------------------------------------------------\n# Header\n# ---------------------------------------------------------------------------\nst.markdown(\"\"\"\n<div class=\"header-box\">\n  <h1 style=\"color:white;margin:0;font-size:32px;letter-spacing:-0.5px;\">\n    🌏 Global Natural Disaster Analysis & Prediction\n  </h1>\n  <p style=\"margin:8px 0 0 0;opacity:0.9;font-size:15px;font-weight:300;\">\n    Interactive dashboard for exploring trends and projections of natural disaster frequencies\n    for the next 5 years (2027–2031) based on Linear Regression &amp; KNN Regression.\n  </p>\n</div>\n\"\"\", unsafe_allow_html=True)\n\n# ---------------------------------------------------------------------------\n# Load data & models\n# ---------------------------------------------------------------------------\nfor path, name in [(DATA_FILE, \"complete_prediction_results.csv\"),\n                   (MODEL_FILE, \"best_models.pkl\"),\n                   (AGG_FILE,   \"processed_agg.csv\")]:\n    if not os.path.exists(path):\n        st.error(f\"❌ File not found: `{path}`\")\n        st.info(\"💡 Please run the preprocessing and model training scripts locally first.\")\n        st.stop()\n\ndf_pred = pd.read_csv(DATA_FILE)\ndf_agg  = pd.read_csv(AGG_FILE)\nwith open(MODEL_FILE, \"rb\") as f:\n    trained_models = pickle.load(f)\n\nGRAN_COL = \"Granularity (years)\" if \"Granularity (years)\" in df_pred.columns else \"Granularity_Best\"\n\n# ---------------------------------------------------------------------------\n# Sidebar\n# ---------------------------------------------------------------------------\nst.sidebar.header(\"🔍 Filters\")\ncountry  = st.sidebar.selectbox(\"Select Country\", sorted(df_pred[\"Country\"].unique()))\ndisaster = st.sidebar.selectbox(\n    \"Select Disaster Type\",\n    sorted(df_pred[df_pred[\"Country\"] == country][\"Disaster Type\"].unique())\n)\nst.sidebar.markdown(\"---\")\nst.sidebar.markdown(\"\"\"\n<div style=\"background:#F8FAFC;padding:14px;border-radius:10px;border:1px solid #E2E8F0;\">\n  <span style=\"font-size:11px;font-weight:700;color:#64748B;\">STUDENT</span>\n  <p style=\"margin:5px 0 2px 0;font-weight:700;color:#0F172A;font-size:13px;\">Bungalunna Nashuha Camelia</p>\n  <span style=\"font-size:12px;color:#64748B;\">IT - Dian Nuswantoro University</span><br>\n  <span style=\"font-size:11px;color:#94A3B8;\">Machine Learning UAS 2025/2026</span>\n</div>\n\"\"\", unsafe_allow_html=True)\n\n# ---------------------------------------------------------------------------\n# Compute 5-year projection\n# ---------------------------------------------------------------------------\nkey = (country, disaster)\nif key not in trained_models:\n    st.error(f\"Model for combination {country} – {disaster} is not available.\")\n    st.stop()\n\ninfo       = trained_models[key]\nmodel      = info[\"model\"]\ng          = info[\"granularity\"]\nmodel_name = info[\"model_name\"]\n\nproj_5yr = {}\nfor year in YEAR_LIST:\n    bin_t = (year // g) * g\n    pred  = max(0, round(model.predict([[bin_t]])[0], 2))\n    proj_5yr[year] = pred\n\nrow_df  = df_pred[(df_pred[\"Country\"] == country) & (df_pred[\"Disaster Type\"] == disaster)]\nmae_lr  = round(float(row_df.iloc[0].get(\"MAE_LR\",  0)), 3) if not row_df.empty else \"-\"\nmae_knn = round(float(row_df.iloc[0].get(\"MAE_KNN\", 0)), 3) if not row_df.empty else \"-\"\n\n# ---------------------------------------------------------------------------\n# Metric cards (4 columns)\n# ---------------------------------------------------------------------------\nst.subheader(f\"📊 Dashboard: {country} — {disaster}\")\n\nc1, c2, c3, c4 = st.columns(4)\nwith c1:\n    st.markdown(f'''\n    <div class=\"metric-card\">\n        <div class=\"metric-icon\">🤖</div>\n        <div class=\"metric-label\">Best Model</div>\n        <div class=\"metric-value\" style=\"color:#6366F1;font-size:16px;\">{model_name}</div>\n    </div>''', unsafe_allow_html=True)\nwith c2:\n    st.markdown(f'''\n    <div class=\"metric-card\">\n        <div class=\"metric-icon\">⏳</div>\n        <div class=\"metric-label\">Temporal Granularity</div>\n        <div class=\"metric-value\" style=\"color:#14B8A6;\">{g} Years</div>\n    </div>''', unsafe_allow_html=True)\nwith c3:\n    st.markdown(f'''\n    <div class=\"metric-card\">\n        <div class=\"metric-icon\">📉</div>\n        <div class=\"metric-label\">MAE Linear Regression</div>\n        <div class=\"metric-value\" style=\"color:#EC4899;\">{mae_lr}</div>\n    </div>''', unsafe_allow_html=True)\nwith c4:\n    st.markdown(f'''\n    <div class=\"metric-card\">\n        <div class=\"metric-icon\">📊</div>\n        <div class=\"metric-label\">MAE KNN Regression</div>\n        <div class=\"metric-value\" style=\"color:#F97316;\">{mae_knn}</div>\n    </div>''', unsafe_allow_html=True)\n\nst.markdown(\"---\")\n\n# ---------------------------------------------------------------------------\n# Chart + Table: 5-year projection\n# ---------------------------------------------------------------------------\nst.subheader(\"📅 Time Series Projection (2027-2031)\")\ncol_chart, col_tabel = st.columns([3, 2])\n\nwith col_chart:\n    fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=\"#FFFFFF\")\n    ax.set_facecolor(\"#FFFFFF\")\n    hist = df_agg[(df_agg[\"Country\"] == country) &\n                  (df_agg[\"Disaster Type\"] == disaster)].sort_values(\"Year\")\n    if not hist.empty:\n        ax.plot(hist[\"Year\"], hist[\"Disaster_Count\"],\n                color=\"#6366F1\", linewidth=2.5, marker=\"o\", markersize=5,\n                markerfacecolor=\"#4F46E5\", markeredgecolor=\"#FFFFFF\",\n                label=\"Historical (1900-2023)\")\n    year_proj = list(proj_5yr.keys())\n    value_proj = list(proj_5yr.values())\n    bars = ax.bar(year_proj, value_proj, color=\"#FB923C\", alpha=0.9,\n                  width=0.55, edgecolor=\"#EA580C\", linewidth=1.2,\n                  label=\"Model Projection (2027-2031)\")\n    for bar in bars:\n        h = bar.get_height()\n        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.03, str(h),\n                ha=\"center\", va=\"bottom\", fontsize=9, fontweight=\"bold\", color=\"#EA580C\")\n    ax.spines[\"top\"].set_visible(False)\n    ax.spines[\"right\"].set_visible(False)\n    ax.spines[\"left\"].set_color(\"#CBD5E1\")\n    ax.spines[\"bottom\"].set_color(\"#CBD5E1\")\n    ax.set_xlabel(\"Year\", fontsize=10, fontweight=\"bold\", color=\"#475569\")\n    ax.set_ylabel(\"Event Count\", fontsize=10, fontweight=\"bold\", color=\"#475569\")\n    ax.legend(frameon=True, facecolor=\"#F8FAFC\", edgecolor=\"none\", fontsize=9)\n    ax.grid(axis=\"y\", linestyle=\"--\", alpha=0.4, color=\"#E2E8F0\")\n    plt.tight_layout()\n    st.pyplot(fig)\n    plt.close()\n\nwith col_tabel:\n    st.markdown(\"**Projection Details Table:**\")\n    tabel = pd.DataFrame({\n        \"Year\": year_proj,\n        \"Predicted Events\": value_proj,\n        \"Average/Year\": [round(v / g, 2) for v in value_proj]\n    })\n    st.dataframe(tabel, use_container_width=True, hide_index=True)\n    diff = value_proj[-1] - value_proj[0]\n    if diff > 0:\n        tren_label, tren_color, tren_text = \"Increasing\", \"#FEF3C7\", \"#92400E\"\n    elif diff < 0:\n        tren_label, tren_color, tren_text = \"Decreasing\", \"#DCFCE7\", \"#166534\"\n    else:\n        tren_label, tren_color, tren_text = \"Stable\", \"#F1F5F9\", \"#475569\"\n    st.markdown(\n        f'<div class=\"trend-box\" style=\"background:{tren_color};\">'\n        f'<span style=\"font-size:11px;font-weight:700;color:#64748B;\">Trend Signal 2027-2031</span>'\n        f'<p style=\"margin:4px 0 0 0;font-size:22px;font-weight:800;color:{tren_text};\">{tren_label}</p>'\n        f'</div>',\n        unsafe_allow_html=True\n    )\n\nst.markdown(\"---\")\n\n# ---------------------------------------------------------------------------\n# Model evaluation section\n# ---------------------------------------------------------------------------\nst.subheader(\"📈 Model Evaluation & Reliability\")\ncol_img, col_desc = st.columns([3, 2])\n\nwith col_img:\n    if os.path.exists(IMAGE_FILE):\n        st.image(IMAGE_FILE,\n                 caption=\"Boxplot MAE, RMSE, R2 — Linear Regression vs KNN\",\n                 use_container_width=True)\n    else:\n        st.warning(\"⚠️ Evaluation chart is not found. Please download `model_comparison.png` from Google Colab or run evaluation locally.\")\n\nwith col_desc:\n    lr_wins  = (df_pred[\"Best_Model\"] == \"Linear Regression\").sum()\n    knn_wins = (df_pred[\"Best_Model\"] == \"KNN Regression\").sum()\n    total    = len(df_pred)\n    st.markdown(f\"\"\"\n<div class=\"info-box\">\n  <b style=\"color:#4F46E5;\">Model Win Statistics ({total} combinations):</b>\n  <ul style=\"margin:8px 0 0 0;padding-left:18px;font-size:14px;color:#475569;line-height:1.7;\">\n    <li><b>Linear Regression</b>: {lr_wins} combinations ({lr_wins/total*100:.1f}%)</li>\n    <li><b>KNN Regression</b>: {knn_wins} combinations ({knn_wins/total*100:.1f}%)</li>\n  </ul>\n</div>\n<p style=\"font-size:13px;color:#64748B;line-height:1.5;\">\n  <b>Interpretation:</b> Linear Regression is generally superior for time-series extrapolation\n  as it models the constant rate of change. KNN is constrained within the range of training inputs (1900-2023).\n  Adaptive granularity (1/2/3/5 years) is selected per country-disaster pair based on the lowest MAE.\n</p>\n\"\"\", unsafe_allow_html=True)\n"""
with open('/content/app_colab.py', 'w') as f:
    f.write(app_code)
print('app_colab.py written successfully.')

# 4. Run Streamlit & Ngrok
import os, subprocess, time
from pyngrok import ngrok
os.system('fuser -k 8501/tcp')
proc = subprocess.Popen(['streamlit', 'run', '/content/app_colab.py', '--server.port', '8501', '--server.headless', 'true'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(8)
if proc.poll() is not None:
    out, err = proc.communicate()
    print('Streamlit crash! Error:'); print(err.decode())
else:
    url = ngrok.connect(8501)
    print(f'Dashboard active at: {url}')

## 📝 Final Conclusions

Based on the experiments conducted:

1. **Trend Confirmed**: Disaster frequency in Indonesia and globally has increased significantly over time. Flood events in Indonesia grew from 0.30 events/year in the 1960s to 2.25 events/year in the 2020s — a ~7.5× increase over six decades, strongly supporting the core hypothesis.

2. **Model Performance**: **Linear Regression** outperforms KNN Regression on 77% of all Country × Disaster Type combinations (376 out of 488), confirming that a linear trend model is appropriate for most historical disaster time series.

3. **Prediction Results**: 488 unique Country × Disaster Type combinations were successfully modeled and predicted for 2027. For Indonesia, Flood is predicted to remain the most frequent disaster type (2.66 events), followed by Storm (1.54) and Earthquake (1.20).

4. **Hyperparameter Tuning**: GridSearchCV on the global trend series confirmed the best KNN configuration. Both models were evaluated on a fully held-out test set that was never seen during training or tuning, ensuring unbiased final performance estimates.

5. **Feature Importance**: Linear Regression coefficients confirm that the time variable (year) has a predominantly positive slope across disaster types — Flood and Storm show the steepest upward trends globally, while a minority of combinations show declining trends reflecting possible improvements in disaster prevention or reporting changes.

6. **Artifact Export**: Full prediction results for all 488 combinations have been saved to `complete_prediction_results.csv` for further use.